In [1]:
import pandas as pd
import numpy as np
from dep2pyodbc import dep2connection

pd.set_option("display.max_columns", None)

channel_crh = dep2connection("CRH")
channel_dwh_lisa = dep2connection("CRH_DWH")
cursor = channel_dwh_lisa.cursor()

pyodbc using windows
pyodbc using windows


In [2]:
sjt = pd.read_csv('./csv/raw_data/decoded_sjt.csv')
sjt.head()

,Unnamed: 0,CandidateID,InstanceID,itemId,sequenceId1,sequenceId2,sequenceId3,answer1,answer2,answer3,score1,score2,score3,timeSpent
0,0,4937367,1,1,1,2,3,5,4,2,0,0,0,181
1,1,4937367,1,2,1,2,3,4,3,2,0,0,0,172
2,2,4937367,1,3,1,2,3,4,2,3,0,0,0,136
3,3,4937367,1,4,1,2,3,3,2,4,0,0,0,266
4,4,4937367,1,5,1,2,3,2,5,1,0,0,0,198


In [3]:
df_sjt = sjt[['CandidateID', 'InstanceID', 'itemId', 'answer1', 'answer2', 'answer3', 'timeSpent']]
df_sjt.rename(columns={
    'itemId':'ItemID', 
    'answer1': 'AnswerSequence1',
    'answer2': 'AnswerSequence2',
    'answer3': 'AnswerSequence3',
    'timeSpent': 'TimeSpent'
    }, inplace=True)
df_sjt['Test'] = 'SJT'
df_sjt.head()

C:\Users\monad\AppData\Local\Temp\ipykernel_12820\629050158.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sjt.rename(columns={
C:\Users\monad\AppData\Local\Temp\ipykernel_12820\629050158.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sjt['Test'] = 'SJT'


,CandidateID,InstanceID,ItemID,AnswerSequence1,AnswerSequence2,AnswerSequence3,TimeSpent,Test
0,4937367,1,1,5,4,2,181,SJT
1,4937367,1,2,4,3,2,172,SJT
2,4937367,1,3,4,2,3,136,SJT
3,4937367,1,4,3,2,4,266,SJT
4,4937367,1,5,2,5,1,198,SJT


### Candidates 

In [4]:
df_candidates_before_key = pd.read_sql("SELECT ID, CandidateID, InstanceID, CreatedDate, ModifiedDate, VersionNumber FROM CandidateResultSJT", channel_crh)
df_candidates_before_key.head()

C:\Users\monad\AppData\Local\Temp\ipykernel_12820\3616229786.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_candidates_before_key = pd.read_sql("SELECT ID, CandidateID, InstanceID, CreatedDate, ModifiedDate, VersionNumber FROM CandidateResultSJT", channel_crh)


KeyboardInterrupt: 

In [ ]:
df_sjt = pd.merge(df_sjt, df_candidates_before_key, on=["CandidateID", "InstanceID"], how="left")
df_sjt.head()

,CandidateID,InstanceID,ItemID,AnswerSequence1,AnswerSequence2,AnswerSequence3,TimeSpent,Test,ID,CreatedDate,ModifiedDate,VersionNumber
0,4937367,1,1,5,4,2,181,SJT,1,2023-11-08 09:01:49.507,2023-11-08 09:41:37.047,v1.0
1,4937367,1,2,4,3,2,172,SJT,1,2023-11-08 09:01:49.507,2023-11-08 09:41:37.047,v1.0
2,4937367,1,3,4,2,3,136,SJT,1,2023-11-08 09:01:49.507,2023-11-08 09:41:37.047,v1.0
3,4937367,1,4,3,2,4,266,SJT,1,2023-11-08 09:01:49.507,2023-11-08 09:41:37.047,v1.0
4,4937367,1,5,2,5,1,198,SJT,1,2023-11-08 09:01:49.507,2023-11-08 09:41:37.047,v1.0


In [ ]:
dim_candidate = pd.read_sql("SELECT CandidateKey, ID, InstanceID FROM DimCandidate", channel_dwh_lisa)
dim_candidate.head()

C:\Users\monad\AppData\Local\Temp\ipykernel_6456\977585254.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_candidate = pd.read_sql("SELECT CandidateKey, ID, InstanceID FROM DimCandidate", channel_dwh_lisa)


,CandidateKey,ID,InstanceID
0,14,1,4
1,22,2,2
2,24,2,4
3,32,3,2
4,34,3,4


In [ ]:
df_sjt = pd.merge(df_sjt, dim_candidate, left_on=["CandidateID", "InstanceID"], right_on=["ID", "InstanceID"], how="left")
df_sjt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89622 entries, 0 to 89621
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   CandidateID      89622 non-null  int64         
 1   InstanceID       89622 non-null  int64         
 2   ItemID           89622 non-null  int64         
 3   AnswerSequence1  89622 non-null  int64         
 4   AnswerSequence2  89622 non-null  int64         
 5   AnswerSequence3  89622 non-null  int64         
 6   TimeSpent        89622 non-null  int64         
 7   Test             89622 non-null  object        
 8   ID_x             89622 non-null  int64         
 9   CreatedDate      89622 non-null  datetime64[ns]
 10  ModifiedDate     89622 non-null  datetime64[ns]
 11  VersionNumber    89622 non-null  object        
 12  CandidateKey     58396 non-null  float64       
 13  ID_y             58396 non-null  float64       
dtypes: datetime64[ns](2), float64(2), int6

In [ ]:
df_sjt.drop(columns=["ID_y", "CandidateID"], inplace=True)
df_sjt.rename(columns={"ID_x": "TestID"}, inplace=True)
df_sjt.head()

,InstanceID,ItemID,AnswerSequence1,AnswerSequence2,AnswerSequence3,TimeSpent,Test,TestID,CreatedDate,ModifiedDate,VersionNumber,CandidateKey
0,1,1,5,4,2,181,SJT,1,2023-11-08 09:01:49.507,2023-11-08 09:41:37.047,v1.0,49373671.0
1,1,2,4,3,2,172,SJT,1,2023-11-08 09:01:49.507,2023-11-08 09:41:37.047,v1.0,49373671.0
2,1,3,4,2,3,136,SJT,1,2023-11-08 09:01:49.507,2023-11-08 09:41:37.047,v1.0,49373671.0
3,1,4,3,2,4,266,SJT,1,2023-11-08 09:01:49.507,2023-11-08 09:41:37.047,v1.0,49373671.0
4,1,5,2,5,1,198,SJT,1,2023-11-08 09:01:49.507,2023-11-08 09:41:37.047,v1.0,49373671.0


### Dates

In [ ]:
df_dates = pd.read_sql("SELECT DateKey, Date FROM DimDate", channel_dwh_lisa)
df_dates.head()

C:\Users\monad\AppData\Local\Temp\ipykernel_6456\594440161.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dates = pd.read_sql("SELECT DateKey, Date FROM DimDate", channel_dwh_lisa)


,DateKey,Date
0,20070101,2007-01-01
1,20070102,2007-01-02
2,20070103,2007-01-03
3,20070104,2007-01-04
4,20070105,2007-01-05


In [ ]:
df_sjt["CreatedDate"] = pd.to_datetime(df_sjt["CreatedDate"]).dt.date
df_sjt = pd.merge(df_sjt, df_dates, left_on="CreatedDate", right_on="Date", how="left")
df_sjt.drop(columns=["Date", "CreatedDate"], inplace=True)
df_sjt.rename(columns={"DateKey": "CreatedDateKey"}, inplace=True)

df_sjt["ModifiedDate"] = pd.to_datetime(df_sjt["ModifiedDate"]).dt.date
df_sjt = pd.merge(df_sjt, df_dates, left_on="ModifiedDate", right_on="Date", how="left")
df_sjt.drop(columns=["Date", "ModifiedDate"], inplace=True)
df_sjt.rename(columns={"DateKey": "ModifiedDateKey"}, inplace=True)

df_sjt.head()

,InstanceID,ItemID,AnswerSequence1,AnswerSequence2,AnswerSequence3,TimeSpent,Test,TestID,VersionNumber,CandidateKey,CreatedDateKey,ModifiedDateKey
0,1,1,5,4,2,181,SJT,1,v1.0,49373671.0,20231108,20231108
1,1,2,4,3,2,172,SJT,1,v1.0,49373671.0,20231108,20231108
2,1,3,4,2,3,136,SJT,1,v1.0,49373671.0,20231108,20231108
3,1,4,3,2,4,266,SJT,1,v1.0,49373671.0,20231108,20231108
4,1,5,2,5,1,198,SJT,1,v1.0,49373671.0,20231108,20231108


### Next TestKey available

In [ ]:
df_test_fca = pd.read_csv('../decoded_data/FCA/FactTest.csv')
df_test_fca.head()

,TestKey,CandidateKey,CreatedDateKey,ModifiedDateKey,PostedDateKey,TestStartTimeKey,TestFinishTimeKey,VersionNumber,Test
0,1,544.0,20220908,20220908,20220908,93154,93212,V1.0,FCA
1,2,854.0,20221207,20221207,20221207,134516,141829,v1.0,FCA
2,3,7564.0,20230809,20230809,20230809,100349,103238,v1.0,FCA
3,4,13914.0,20231103,20231103,20231103,95141,104332,v1.0,FCA
4,5,14244.0,20231025,20231025,20231025,62301,71219,v1.0,FCA


In [ ]:
max_key = df_test_fca.TestKey.max()
max_key

159143

## FactTest

In [ ]:
df_test = df_sjt[["TestID", "Test", "CandidateKey", "CreatedDateKey", "VersionNumber"]]

df_test.drop_duplicates(inplace=True)
df_test.reset_index(inplace=True, drop=True)
df_test["TestKey"] = df_test.index + max_key + 1

df_test.head()

C:\Users\monad\AppData\Local\Temp\ipykernel_6456\287193157.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.drop_duplicates(inplace=True)
C:\Users\monad\AppData\Local\Temp\ipykernel_6456\287193157.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["TestKey"] = df_test.index + max_key + 1


,TestID,Test,CandidateKey,CreatedDateKey,VersionNumber,TestKey
0,1,SJT,49373671.0,20231108,v1.0,159144
1,2,SJT,49546791.0,20231207,v1.0,159145
2,3,SJT,NaN,20171221,v1.0,159146
3,4,SJT,NaN,20181009,v1.0,159147
4,5,SJT,47543331.0,20230119,V0.1,159148


## FactQuestionSJT

In [ ]:
df_question = df_sjt.drop(columns=["CandidateKey", "CreatedDateKey", "ModifiedDateKey", "VersionNumber"])
df_question["QuestionKey"] = df_question.index + 1
df_question.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89622 entries, 0 to 89621
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   InstanceID       89622 non-null  int64 
 1   ItemID           89622 non-null  int64 
 2   AnswerSequence1  89622 non-null  int64 
 3   AnswerSequence2  89622 non-null  int64 
 4   AnswerSequence3  89622 non-null  int64 
 5   TimeSpent        89622 non-null  int64 
 6   Test             89622 non-null  object
 7   TestID           89622 non-null  int64 
 8   QuestionKey      89622 non-null  int64 
dtypes: int64(8), object(1)
memory usage: 6.2+ MB


In [ ]:
df_question = pd.merge(df_question, df_test[["TestKey", "TestID"]], on="TestID", how="left")
df_question.set_index('QuestionKey', inplace=True)
df_question.head()

,InstanceID,ItemID,AnswerSequence1,AnswerSequence2,AnswerSequence3,TimeSpent,Test,TestID,TestKey
QuestionKey,,,,,,,,,
1,1,1,5,4,2,181,SJT,1,159144
2,1,2,4,3,2,172,SJT,1,159144
3,1,3,4,2,3,136,SJT,1,159144
4,1,4,3,2,4,266,SJT,1,159144
5,1,5,2,5,1,198,SJT,1,159144


In [ ]:
df_test.drop(columns=["TestID"], inplace=True)
df_question.drop(columns=["TestID"], inplace=True)

C:\Users\monad\AppData\Local\Temp\ipykernel_6456\525301613.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.drop(columns=["TestID"], inplace=True)


## TimeSpent Test

In [ ]:
df = df_question[["TestKey", "TimeSpent"]]

df_timespent = df.groupby('TestKey')['TimeSpent'].sum().reset_index()
df_timespent.head()

,TestKey,TimeSpent
0,159144,2915
1,159145,2048
2,159146,0
3,159147,0
4,159148,3962


In [ ]:
df_test = df_test.merge(df_timespent, on='TestKey')
df_test

,Test,CandidateKey,CreatedDateKey,VersionNumber,TestKey,TimeSpent
0,SJT,49373671.0,20231108,v1.0,159144,2915
1,SJT,49546791.0,20231207,v1.0,159145,2048
2,SJT,NaN,20171221,v1.0,159146,0
3,SJT,NaN,20181009,v1.0,159147,0
4,SJT,47543331.0,20230119,V0.1,159148,3962
...,...,...,...,...,...,...
6889,SJT,3869263.0,20240717,v1.0,166033,0
6890,SJT,3869463.0,20240717,v1.0,166034,0
6891,SJT,3869513.0,20240717,v1.0,166035,0
6892,SJT,44501345.0,20220525,v1.0,166036,0


## To csv 

In [ ]:
df_test.to_csv('../decoded_data/SJT/FactTest.csv', index=False)
df_question.to_csv('../decoded_data/SJT/FactQuestionSJT.csv')